In [ ]:
# Setup: clone the repo and switch to the `cmkd` branch (all CMKD code lives there, not on main)
!git clone https://github.com/TotallyNotMinh/Environmental-Sound-Recognition-Under-Unseen-Conditions.git
%cd /kaggle/working/Environmental-Sound-Recognition-Under-Unseen-Conditions
!git fetch origin
!git checkout cmkd
!git pull origin cmkd

In [ ]:
!pip install -q h5py scipy opencv-python tqdm

In [ ]:
# Quick checks (~a few minutes, no dataset or teacher needed). Run these before the long jobs below.
# 1) KD loss sanity checks -> should print "KDLoss sanity checks passed."
!python losses/distillation.py
# 2) End-to-end smoke tests on synthetic data (single GPU, 2 epochs each)
!python scripts/train_cnn_classifier.py --mock --num-epoch 2 --num-workers 0 --checkpoint-dir /kaggle/working/smoke/cnn/
!python scripts/train_kd.py --mock --num-epoch 2 --num-workers 0 --checkpoint-dir /kaggle/working/smoke/kd/

In [ ]:
# STEP 1 -- Train the CNN teacher (EfficientNet-B0) WITHOUT KD.
# CMKD FSD50K CNN recipe: LR 5e-4, batch 24, 50 epochs, BCE.
# --batch-size is the TOTAL batch; with 2 GPUs each GPU gets 12.
# Output used by STEP 2: /kaggle/working/checkpoints/fsd50k_cnn_b0/best_cnn.pth
!torchrun --nproc_per_node=2 scripts/train_cnn_classifier.py \
        --data-path /kaggle/input/datasets/yousirui1/fsd50k/fsd50k \
        --batch-size 24 \
        --lr 5e-4 \
        --num-epoch 50 \
        --checkpoint-dir /kaggle/working/checkpoints/fsd50k_cnn_b0/

In [ ]:
# STEP 2 -- Distill the frozen CNN teacher into an AST-Base student (run after STEP 1 finishes).
# CMKD FSD50K AST recipe: LR 5e-5, batch 12, 50 epochs, BCE + KD (lambda 0.5, tau 1.0).
# --batch-size is the TOTAL batch; with 2 GPUs each GPU gets 6.
!test -f /kaggle/working/checkpoints/fsd50k_cnn_b0/best_cnn.pth || echo "ERROR: teacher checkpoint missing -- run STEP 1 first"
!torchrun --nproc_per_node=2 scripts/train_kd.py \
        --data-path /kaggle/input/datasets/yousirui1/fsd50k/fsd50k \
        --teacher-checkpoint /kaggle/working/checkpoints/fsd50k_cnn_b0/best_cnn.pth \
        --batch-size 12 \
        --lr 5e-5 \
        --num-epoch 50 \
        --checkpoint-dir /kaggle/working/checkpoints/fsd50k_kd_ast_base/